# Chess Representation Study @ NeurIPS 2026 — CHECK MODE (tiny, raises on failure)

Gemma 4 E2B/E4B (local, 4-bit) vs DeepSeek V4 Flash (gateway API) on mate-in-1/2 + best-move tasks under grid/FEN/bitboard/list representations (see README).
Results land in `results_check/` and are zipped for download.
- Positions + exact oracles: committed (`data/positions/`), built from the lichess puzzle DB + eval DB (CC0).
- Engine + dataset tests gate every run: `scripts/test_engine.py`.
- Sweep: `scripts/run_suite.py` (3 models x tasks x reps).

## 1. Get the repo (GitHub secret method)

The repo is **private**. On Kaggle, a secret reaches the notebook ONLY if it is **attached to this notebook** and the kernel is started AFTER attaching:

1. Notebook editor -> **+ Add** (top-right) -> **Add secret** -> select `GITHUB_TOKEN`
2. **Save** the notebook (Ctrl+S).
3. **Kernel -> Restart & Run All** (env vars are injected at kernel start; plain "Run All" does NOT pick up newly attached secrets).

Required secrets: `GITHUB_TOKEN` (repo clone + live uploads), `HF_TOKEN` (gated gemma-4 checkpoints), `OPENCODE_API_KEY` (deepseek-v4-flash gateway).
This cell reads the token from the env var, and falls back to Kaggle's own `kaggle_secrets` API if the env var is missing.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "chess-slm-benchmark"
# ALWAYS start from a fresh clone: re-runs in the same Kaggle session keep the
# old /kaggle/working repo, and stale code has bitten us more than once.
if REPO.exists():
    shutil.rmtree(REPO)

def find_token():
    for name in ("GITHUB_TOKEN", "GH_TOKEN"):
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

# diagnostic: what token-ish env vars are actually present?
present = sorted(k for k in os.environ if "TOKEN" in k.upper() or "SECRET" in k.upper())
print("token-ish env vars present:", present, flush=True)
token = find_token()
print("GITHUB_TOKEN resolved:", bool(token), flush=True)
url = "https://github.com/Vedang-P/chess-slm-benchmark.git"
if token:
    url = url.replace("https://", f"https://x-access-token:{token}@")
res = subprocess.run(["git", "clone", "--quiet", url, str(REPO)],
                     capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError(
        "git clone failed. The token did not reach this run. Fix order: "
        "(1) + Add -> Add secret -> GITHUB_TOKEN; (2) SAVE the notebook; "
        "(3) Kernel -> Restart & Run All. Then check the diagnostic line above: "
        "if 'token-ish env vars present' is empty, the secret is not attached to "
        "THIS notebook. Stderr: " + res.stderr[-300:]
    )
os.chdir(REPO)
print("cwd:", Path.cwd())

## 2. Dependencies (forced upgrade; transformers must be >= 5.13 for Gemma 4)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-U", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "wandb"], check=True)
import transformers
if int(transformers.__version__.split(".")[0]) < 5:
    raise RuntimeError(
        f"transformers {transformers.__version__} is too old for Gemma 4 "
        "(needs >= 5.13). The upgrade failed — check the pip install output above."
    )
print("transformers", transformers.__version__, "(gemma4 support OK)")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Stage runner (never raises; the verdict cell checks results)

In [ ]:
import json, time, shutil
from pathlib import Path

STAGE_LOG = Path("results_check/stage_log.json")
def run_stage(name, args, timeout_min):
    Path("results_check").mkdir(parents=True, exist_ok=True)
    rec = {"stage": name, "status": "running", "elapsed_min": None}
    t0 = time.time()
    try:
        res = subprocess.run(args, timeout=timeout_min * 60)
        rec["status"] = "ok" if res.returncode == 0 else "failed"
        rec["returncode"] = res.returncode
    except subprocess.TimeoutExpired:
        rec["status"] = "timeout"
    except Exception as e:
        rec["status"] = "error"
        rec["error"] = str(e)[:200]
    rec["elapsed_min"] = round((time.time() - t0) / 60, 1)
    entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
    entries.append(rec)
    STAGE_LOG.write_text(json.dumps(entries, indent=1))
    print(f"stage {name}: {rec['status']} ({rec['elapsed_min']}min)", flush=True)
    return rec["status"]

## 4. Gate: engine + dataset tests

In [ ]:
status = run_stage("engine_tests", [sys.executable, "scripts/test_engine.py", "--quick"], 10)
if status != "ok":
    raise RuntimeError("engine tests failed -- see output above")

## 5. Data validation (committed task sets + oracle fields)

In [ ]:
import json
for name in ["mate1-lichess", "mate2-lichess", "bestmove-8x8", "mate-selection-test"]:
    recs = json.loads(Path(f"data/positions/{name}.json").read_text())
    assert len(recs) >= 40, f"{name}: expected >=40 positions, got {len(recs)}"
    assert all("win_moves" in r and "lose_moves" in r for r in recs)
print("core task data OK (3 task sets, oracle fields present)")

## 6. Recover completed results (after a died session)

Every completed cell's summary + samples are backed up to the public live repo by `--monitor` (check and full results live in separate namespaces). If this is a fresh session, pull them back so `--resume` can skip what already ran.

In [ ]:
import json, urllib.request
from pathlib import Path

out = Path("results_check/chess")
out.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/Vedang-P/chess-bench-live/main"
idx_url = f"{base}/results_check/index.json"
if list(out.glob("*.summary.json")):
    print("results already present locally")
else:
    try:
        idx = json.load(urllib.request.urlopen(idx_url, timeout=20))
        for name in idx["files"]:
            url = f"{base}/results_check/chess/{name}"
            (out / name).write_bytes(urllib.request.urlopen(url, timeout=20).read())
        print(f"recovered {len(idx['files'])} completed summaries from the live repo")
    except Exception as e:
        print("nothing to recover (first run or no backup yet):", e)

## 7. The sweep (3 models x tasks x reps)

`--monitor` uploads state.json every 60s and one batched upload per completed cell (summary + samples + index) — never per-sample. `--resume` skips cells whose schema-current summary already exists (recovered in the previous cell).

In [ ]:
sweep_args = [sys.executable, "scripts/run_suite.py", "--output_dir", "results_check/chess",
              "--monitor", "--monitor-interval", "60"]
if True:
    sweep_args.append("--check")
sweep_args.append("--resume")   # skip cells whose summaries were recovered
if True:
    # check mode = full visibility: print the exact prompt, stream each
    # model's output live, and add 'think step by step' so even non-reasoning
    # models show their reasoning
    sweep_args += ["--verbose", "--stream", "--cot"]
status = run_stage("chess_sweep", sweep_args, 100)
print("sweep:", status)

## 8. Results table

In [ ]:
import pandas as pd
csv_path = Path("results_check/chess/comparison_table.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df)
    print("rows:", len(df))
else:
    print("no comparison table -- sweep did not complete")

## 9. Verdict (check mode: fail loudly)

In [ ]:
entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
fails = [e for e in entries if e["status"] != "ok"]
if fails:
    raise RuntimeError(f"check mode: {len(fails)} failed stages: {[e['stage'] for e in fails]}")
print("ALL CHECK STAGES PASSED")

## Notes
- **Secrets:** `GITHUB_TOKEN`, `HF_TOKEN` (gemma gated), `OPENCODE_API_KEY` (deepseek-v4-flash). All three must be attached + kernel restarted.
- **Resume after a died session:** re-run the notebook; the recovery cell pulls backed-up summaries + samples, and `--resume` skips them.
- **Monitoring is batched:** one contents-API upload per completed cell + state.json every 60s. The dashboard shows <=60s lag. GitHub is never hammered with per-sample commits.
- **Timeouts:** full-mode sweep is capped at 12h. DeepSeek runs the full matrix (~2s/position); gemma dominates the wall-clock.